In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from data_processing.loading.dataframe_loading import load_psd
from data_processing.arc_paths import get_parq_root, get_report_root, INPUT_DATA_FOLDER
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram, 
    scan_histogram_slices
)
from data_processing.reporting.plotting import plot_psd_histogram, plot_scatter
from data_processing.dataframe_validation import get_df_col, DataframeColumn

from scipy.signal import savgol_filter, find_peaks, find_peaks_cwt
from scipy.interpolate import UnivariateSpline

In [ ]:
# Data location

experiment_name = 'ID-76'
experiment_display_name = 'EJ-309 Magnetic Interference Testing - KOH present, inside reactor chamber, with shield box and metal wall, thruster present'
experiment_name_2 = 'ID-79'
experiment_display_name = 'EJ-309 Magnetic Interference Testing - KOH present, inside reactor chamber, with shield box and metal wall, thruster absent'

In [ ]:
# Locations
PARQ_ROOT = get_parq_root(experiment_name)
REPORT_ROOT = get_report_root(experiment_name)

In [ ]:
# read first csv into dataframe
# get header from first csv
psd_report = load_psd(experiment_name)

In [ ]:
psd_report.head()

In [ ]:
energy_spectrum_path = INPUT_DATA_FOLDER / experiment_name / "processed_data/unfiltered/spectra/energy_spectrum_0.parquet"

In [ ]:
energy_spectrum = pd.read_parquet(energy_spectrum_path)
energy_spectrum

In [ ]:
energy_channel_array = energy_spectrum['channel']
bin_energy_array = energy_spectrum['bin_energy']

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(energy_channel_array, energy_spectrum['count'])

In [ ]:
energy_spectrum_smooth = savgol_filter(energy_spectrum['count'], 51, 7)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(energy_channel_array, energy_spectrum_smooth)

In [ ]:
# peak finding
peaks, properties = find_peaks(energy_spectrum_smooth, prominence=4, width=100)
print(peaks)
print(properties['prominences'])
print(properties['widths'])

In [ ]:
k_peak = peaks[-1]
k_peak_energy = bin_energy_array[k_peak]

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(bin_energy_array, energy_spectrum_smooth)
ax.axvline(k_peak_energy, dashes=[4,2])
ax.text(k_peak_energy + 0.02, 1500, f"K peak: {k_peak_energy} MeVee")

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(bin_energy_array, energy_spectrum['count'])
ax.plot(bin_energy_array, energy_spectrum_smooth)
# ax.axvline(k_peak_energy, dashes=[4,2])
# ax.text(k_peak_energy + 0.02, 1500, f"K peak: {k_peak_energy} MeVee")

In [ ]:
cwt_peaks = find_peaks_cwt(energy_spectrum['count'], widths=400)
print(cwt_peaks)
print([bin_energy_array[x] for x in cwt_peaks])

In [ ]:
k_peak_cwt = cwt_peaks[-1]
k_peak_cwt_energy = bin_energy_array[k_peak_cwt]

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(bin_energy_array, energy_spectrum_smooth)
ax.axvline(k_peak_energy, dashes=[4,2])
ax.text(k_peak_energy + 0.02, 1500, f"K peak (Normal)")
ax.text(k_peak_energy + 0.02, 1450, f"{k_peak_energy} MeVee")
ax.axvline(k_peak_cwt_energy, dashes=[2,2])
ax.text(k_peak_cwt_energy - 0.02, 1300, f"K peak (CWT method)", horizontalalignment='right')
ax.text(k_peak_cwt_energy - 0.02, 1250, f"{k_peak_cwt_energy} MeVee", horizontalalignment='right')

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(bin_energy_array, energy_spectrum['count'])
ax.plot(bin_energy_array, energy_spectrum_smooth)
ax.axvline(k_peak_energy, dashes=[4,2])
ax.text(k_peak_energy + 0.02, 1500, f"K peak (Normal)")
ax.text(k_peak_energy + 0.02, 1450, f"{k_peak_energy} MeVee")
ax.axvline(k_peak_cwt_energy, dashes=[2,2])
ax.text(k_peak_cwt_energy - 0.02, 1300, f"K peak (CWT method)", horizontalalignment='right')
ax.text(k_peak_cwt_energy - 0.02, 1250, f"{k_peak_cwt_energy} MeVee", horizontalalignment='right')

In [ ]:
all_exp_details = {
    "ID-76": "KOH, inside, with shield + wall plate, thruster present",
    "ID-79": "KOH, inside, with shield + wall plate, thruster absent",
}
experiment_spectra_paths = {exp_name:INPUT_DATA_FOLDER / exp_name / "processed_data/unfiltered/spectra/energy_spectrum_0.parquet" 
                            for exp_name in all_exp_details}

In [ ]:
all_energy_spectra = {exp_name: pd.read_parquet(spectra_path) for exp_name, spectra_path in experiment_spectra_paths.items()}

In [ ]:
plot_data = {}
for exp_name, energy_spectrum in all_energy_spectra.items():
    bin_energy_array = energy_spectrum['bin_energy']
    energy_spectrum_smooth = savgol_filter(energy_spectrum['count'], 51, 7)
#     energy_spectrum_spline = UnivariateSpline(bin_energy_array, energy_spectrum_smooth)
#     energy_spectrum_spline_deriv = energy_spectrum_spline.derivative()
    peaks, _ = find_peaks(energy_spectrum_smooth, width=100, prominence=4)
    print(f"{exp_name}: Peaks at indices {peaks}, corresponding to energies {[bin_energy_array[peak] for peak in peaks]}")
    k_peak = bin_energy_array[peaks[-1]]
    peaks = find_peaks_cwt(energy_spectrum_smooth, widths=400)
    print(f"{exp_name}: CWT peaks at indices {peaks}, corresponding to energies {[bin_energy_array[peak] for peak in peaks]}")
    k_peak_cwt = bin_energy_array[peaks[-1]]
    plot_data[exp_name] = {
        "x": bin_energy_array,
        "y": energy_spectrum_smooth,
        "k_peak": k_peak,
        "k_peak_cwt": k_peak_cwt
    }

In [ ]:
for exp_id, details in all_exp_details.items():
    plot_data[exp_id]['exp_description'] = details

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
for i, (exp_name, plot_series) in enumerate(plot_data.items()):
    ax.plot(plot_series['x'], plot_series['y'], label=f"{exp_name}: {plot_series['exp_description']}")
    k_peak_cwt = plot_series['k_peak_cwt']
    y_pos = 400+i*60
    ax.vlines([k_peak_cwt], linestyle="dashed", ymin=0, ymax=y_pos)
    ax.text(k_peak_cwt + 0.02, y_pos, f"{exp_name}: {k_peak_cwt} MeVee")
ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
for i, (exp_name, plot_series) in enumerate(plot_data.items()):
    ax.plot(plot_series['x'], plot_series['y'], label=f"{exp_name}: {plot_series['exp_description']}")
    k_peak = plot_series['k_peak']
    y_pos = 400+i*60
    ax.vlines([k_peak], linestyle="dashed", ymin=0, ymax=y_pos)
    ax.text(k_peak_cwt + 0.02, y_pos, f"{exp_name}: {k_peak} MeVee")
ax.legend()

In [ ]:
energy_spectrum_deriv = np.gradient(energy_spectrum_smooth)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(energy_channel_array[200:4000], energy_spectrum_deriv[200:4000])

In [ ]:
energy_spectrum_spline = UnivariateSpline(bin_energy_array, energy_spectrum_smooth)
energy_spectrum_spline_array = energy_spectrum_spline(bin_energy_array)
energy_spectrum_spline_deriv = energy_spectrum_spline.derivative()
energy_spectrum_spline_deriv_array = energy_spectrum_spline_deriv(bin_energy_array)

In [ ]:
plt.plot(bin_energy_array, energy_spectrum_spline_array)

In [ ]:
# peak finding
peaks, properties = find_peaks(-energy_spectrum_spline_deriv(bin_energy_array))
peaks

In [ ]:
# peak finding
peaks, properties = find_peaks(-energy_spectrum_spline_deriv(bin_energy_array), width=100, prominence=4)
print(peaks)
print(properties['prominences'])
print(properties['widths'])

In [ ]:
compton_edge = bin_energy_array[peaks[-1]]

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(bin_energy_array[500:], energy_spectrum_spline_deriv_array[500:])
ax.axvline(compton_edge, dashes=[4,2])
ax.text(compton_edge + 0.01, 100, f"Compton Edge: {compton_edge} MeVee")

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(bin_energy_array, energy_spectrum_spline_array)
ax.axvline(compton_edge, dashes=[4,2])
ax.text(compton_edge + 0.02, 250, f"Compton Edge: {compton_edge} MeVee")

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(bin_energy_array, energy_spectrum_spline_array)
ax.axvline(compton_edge, dashes=[4,2])
ax.text(compton_edge + 0.005, 30, f"Compton Edge: {compton_edge} MeVee")
ax.set_xlim(1.0, 1.5)
ax.set_ylim(0, 40)

In [ ]:
energy_bin_width = 0.05

In [ ]:
# Z, xe, ye = get_psd_energy_histogram(psd_report)
x = get_df_col(psd_report, DataframeColumn.CALIB_ENERGY)
y = get_df_col(psd_report, DataframeColumn.PSD)

max_energy = x.max()
print(max_energy)
energy_bins = np.arange(0, max_energy, energy_bin_width)
print(energy_bins)

Z, xe, ye = np.histogram2d(x, y, [energy_bins, 512])

In [ ]:
plot_psd_histogram(psd_report, colorbar=True)

In [ ]:
plot_scatter(psd_report["CALIB_ENERGY"], psd_report["tail / total"])

In [ ]:
%%time
# Scan 2D histogram's energy slices and get bimodal fit

# Settings
start_scan_idx = 0
end_scan_idx = 420



end_scan_idx = min(end_scan_idx, len(Z))
psd_bin_lbs = ye[:-1]

# Default
default_bounds = (
    (0.1, 0.01, 1, 
     0.25, 0.01, 0),
    (0.2, 0.1, Z.max(),
     0.38, 0.04, 2000)
)

bounds_a = (
    (0.1, 0.01, 1, 
     0.35, 0.01, 0),
    (0.2, 0.1, Z.max(),
     0.36, 0.04, 2000)
)

bounds_b = (
    (0.1, 0.01, 1, 
     0.34, 0.01, 0),
    (0.2, 0.1, Z.max(),
     0.36, 0.03, 2000)
)


# Ranged Example
bounds = [
    ((0,60), bounds_a),
]

# df_fom = find_threshold_fom_slice(psd_bin_lbs, Z.T, bounds, 0, end_scan_idx)

df, df_err = scan_histogram_slices(
    psd_bin_lbs, 
    Z.T,
    bounds=bounds,
    default_bounds=default_bounds, 
    start_idx = start_scan_idx, 
    end_idx = end_scan_idx
)

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(df['i'], df['mu1'])
# ax.plot(df['i'], df['sigma1'])

In [ ]:
usable_mu1 = df.query("i >= 10").query("i < 35")['mu1']
mu1_avg = usable_mu1.mean()
mu1_avg

In [ ]:
psd_slice_index = np.where(ye > mu1_avg)[0][0]
psd_slice_index

In [ ]:
gamma_midpoint_slice = Z.T[psd_slice_index]
energy_axis = xe[:-1]

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(energy_axis, gamma_midpoint_slice)

In [ ]:
gamma_slice_smooth = savgol_filter(gamma_midpoint_slice, 11, 3)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(energy_axis, gamma_slice_smooth)

In [ ]:
energy_histo = Z.max(axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(energy_axis, energy_histo)

In [ ]:
energy_histo_smooth = savgol_filter(energy_histo, 11, 3)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(energy_axis, energy_histo_smooth)

In [ ]:
fft_spectrum = np.fft.fft(gamma_midpoint_slice)
plt.plot(energy_axis, fft_spectrum)

In [ ]:
power_spectrum = np.abs(np.fft.fftshift(np.fft.fft(gamma_midpoint_slice)))**2

In [ ]:
fourier_energy_axis = energy_axis - energy_axis[energy_axis.shape[0]//2]
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(fourier_energy_axis, power_spectrum)

In [ ]:
Z_psd_distro, edge_psd_distro = np.histogram(psd_report["tail / total"], bins=energy_bins)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(edge_psd_distro[:-1], Z_psd_distro)